# DNSMOS Pro quality analysis for all audio

This notebook scores the original, vowel-edit, and Praat groups with the
NISQA-trained DNSMOS Pro checkpoint. DNSMOS Pro predicts speech-quality mean
opinion score (MOS); it is not a speech-recognition intelligibility classifier.
The model revision and preprocessing are pinned for reproducibility.

In [1]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display

from dnsmos_common import (
    DNSMOSPRO_COMMIT, build_group_inventory, find_project_root,
    score_all_groups,
)

PROJECT_ROOT = find_project_root()
DEVICE = os.environ.get("DNSMOS_DEVICE", "auto")
FORCE_RECOMPUTE = os.environ.get("DNSMOS_FORCE_RECOMPUTE", "false").lower() in {"1", "true", "yes"}
print(f"Project root: {PROJECT_ROOT}")
print(f"DNSMOSPro commit: {DNSMOSPRO_COMMIT}")
print(f"Device request: {DEVICE}")

Project root: C:\projects\ProMoNet
DNSMOSPro commit: 72f0fa4f71a41e70f12718be214665b1bf4fcbec
Device request: auto


## 1. Inspect the strict three-group inventory

The notebook stops if the expected 150/149/99 structure changes, so files
cannot silently enter or leave the comparison.

In [2]:
inventory = build_group_inventory(PROJECT_ROOT)
counts = inventory.groupby(["pipeline", "condition"]).size().rename("wav_count")
display(counts.to_frame())
display(inventory.head())

wav_count
pipeline            condition                 
original            original               150
praat_pipeline      neu_to_hap              49
                    neu_to_sad              50
vowel_edit_pipeline neu_reconstruct         50
                    neu_to_hap              49
                    neu_to_sad              50

,audio_id,relative_path,filename,file_size_bytes,modified_time_utc,_path,pipeline,condition,speaker,sentence_id,source_emotion,donor_emotion,feature_set
0,47_01_hap_f,audio/47_01_hap_f.wav,47_01_hap_f.wav,157628,2026-06-02T15:02:12.250970+00:00,C:\projects\ProMoNet\audio\47_01_hap_f.wav,original,original,47,01,hap,,natural
1,47_01_neu_f,audio/47_01_neu_f.wav,47_01_neu_f.wav,163466,2026-06-02T15:02:12.266544+00:00,C:\projects\ProMoNet\audio\47_01_neu_f.wav,original,original,47,01,neu,,natural
2,47_01_sad_f,audio/47_01_sad_f.wav,47_01_sad_f.wav,205916,2026-06-02T15:02:12.290614+00:00,C:\projects\ProMoNet\audio\47_01_sad_f.wav,original,original,47,01,sad,,natural
3,47_02_hap_f,audio/47_02_hap_f.wav,47_02_hap_f.wav,139754,2026-06-02T15:02:12.337097+00:00,C:\projects\ProMoNet\audio\47_02_hap_f.wav,original,original,47,02,hap,,natural
4,47_02_neu_f,audio/47_02_neu_f.wav,47_02_neu_f.wav,160674,2026-06-02T15:02:12.345872+00:00,C:\projects\ProMoNet\audio\47_02_neu_f.wav,original,original,47,02,neu,,natural


## 2. Score all recordings

Audio is converted to mono 16 kHz and repetitively cropped to 10 seconds,
matching DNSMOSPro's dataset preprocessing. Existing successful rows are reused
when the file and pinned model revision are unchanged.

In [3]:
scored_groups, average_table = score_all_groups(
    PROJECT_ROOT, device=DEVICE, force_recompute=FORCE_RECOMPUTE
)
for name, frame in scored_groups.items():
    print(f"{name}: {len(frame)} successful rows")
display(average_table.style.format("{:.3f}", na_rep="—"))

original_audio_scores.csv: 150 selected, 150 reused, 0 requiring inference on cpu
vowel_edit_pipeline_scores.csv: 149 selected, 149 reused, 0 requiring inference on cpu
praat_pipeline_scores.csv: 99 selected, 99 reused, 0 requiring inference on cpu


original: 150 successful rows
vowel_edit_pipeline: 149 successful rows
praat_pipeline: 99 successful rows


,neutral,happy,sad
pipeline,,,
original,2.381,2.718,2.563
vowel_edit_pipeline,2.120,2.169,2.144
praat_pipeline,—,2.087,2.123


## 3. Validate saved outputs

The table has pipeline rows and Neutral/Happy/Sad columns. Praat has no neutral
reconstruction, so that cell is intentionally empty.

In [4]:
output_dir = PROJECT_ROOT / "outputs" / "dnsmos"
expected = {
    "original_audio_scores.csv": 150,
    "vowel_edit_pipeline_scores.csv": 149,
    "praat_pipeline_scores.csv": 99,
}
for filename, row_count in expected.items():
    saved = pd.read_csv(output_dir / filename)
    assert len(saved) == row_count
    assert saved["status"].eq("success").all()
summary = pd.read_csv(output_dir / "average_scores_by_pipeline_emotion.csv")
assert summary.shape == (3, 4)
print("Acceptance checks passed for all three DNSMOS groups.")
display(summary.style.format(
    {c: "{:.3f}" for c in ["neutral", "happy", "sad"]}, na_rep="—"
))

Acceptance checks passed for all three DNSMOS groups.

,pipeline,neutral,happy,sad
0,original,2.381,2.718,2.563
1,vowel_edit_pipeline,2.120,2.169,2.144
2,praat_pipeline,—,2.087,2.123
